<a href="https://colab.research.google.com/github/brasilemnumeros/dados/blob/main/indicadores/mvgr/MVG-R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MVG-R — nominal e real

Este notebook é o motor de coleta e publicação das séries `mvgr-nominal` e `mvgr-real`.

- **Fonte:** Banco Central do Brasil — Sistema de Informações de Crédito (SCR), via API de
  Dados Abertos (SGS), série **25419**: "Mediana dos Valores de Garantia de Imóveis
  Residenciais Financiados (MVG-R)", em reais.
- **Endpoint:** `https://api.bcb.gov.br/dados/serie/bcdata.sgs.25419/dados?formato=json`
- **Definição:** mediana dos **valores de avaliação** dos imóveis residenciais dados em
  garantia a financiamentos imobiliários para pessoas físicas (alienação fiduciária e
  hipoteca residencial), informados pelos bancos ao SCR. Janela móvel de **três meses** (o
  mês `M` usa `M`, `M−1` e `M−2`), abrangência nacional, valores **expressos em reais**;
  entram apenas financiamentos com valor contratado acima de 13,3 salários mínimos mensais.
  É um **nível de valor de garantia** — a mediana do valor de avaliação de um imóvel
  financiado —, não um preço de mercado, um índice de preços, uma massa ou um estoque
  agregado de crédito. O piso de 13,3 salários mínimos mensais equivale a um salário mínimo
  anual: é um piso baixo (exclui operações minúsculas), não um corte de alta renda.
- **Fórmula:** nenhuma para o nominal: a série é publicada como o BCB a divulga. O real é
  derivação: `nominal(t) / deflator(t)`, com `deflator(t) = ∏(1 + média_IPCA_s/100)` para
  `s` de `2004-04` a `t`, onde `média_IPCA_s` é a **média geométrica dos fatores mensais do
  IPCA de `s-2`, `s-1` e `s`** — a mesma janela móvel de 3 meses do MVG-R. IPCA = SGS
  **433**, **base fixa em `2004-03`** (reais de março de 2004).
**Nome.** O nome oficial da fonte é **MVG-R** (Mediana dos Valores de Garantia de Imóveis
Residenciais Financiados). O nominal é o **MVG-R**; o real é o **MVG-R deflacionado pelo
IPCA**, não um índice de preços nem o "valor de mercado do imóvel". A mediana é do valor de
avaliação da garantia de imóveis financiados, não do preço de compra de todos os imóveis.

- **Período publicado:** de `2004-03-01` — primeiro mês da série na fonte — até o último mês
  disponível no SGS. A fonte divulga com defasagem de cerca de três meses.
- **Arquivos gerados:** `indicadores/mvgr/mvgr-nominal.json` e
  `indicadores/mvgr/mvgr-real.json`, ambos com `versao: 1`.

## Decisão metodológica

**Nominal — adotar a fonte oficial.** O MVG-R é publicado pelo BCB e a SGS 25419 é a série
primária; o notebook não recalcula, não agrega e não soma nada. Reexecutar a série inteira a
cada rodada é o que mantém o arquivo colado na fonte — a série é revisada pelo BCB quando o
SCR recebe informação atrasada.

**Real — derivação declarada.** O BCB não publica o MVG-R deflacionado: o IVG-R é índice de
preços, não valor em reais. A deflação usa o **IPCA** (SGS 433), coerente com o catálogo do
app ("descontada a inflação (IPCA)") e com o `ipca-acum-12m` já publicado neste repo. O
IGP-M foi descartado por medir preços no atacado e reajustes contratuais, não o custo de vida.
O deflator é aplicado **mês a mês, por composição** — `∏(1 + i/100)` —, onde `i` é a
**média geométrica dos fatores mensais do IPCA da janela de 3 meses** (`M−2`, `M−1`, `M`), a
mesma janela da mediana móvel do MVG-R. Nunca por soma e nunca por interpolação.
Como controle externo, a cadeia do 433 reproduz o número-índice do
IPCA do IBGE (Sidra 1737, variável 2266) com desvio relativo máximo de `2,1 × 10⁻⁵`.

**Base fixa em `2004-03`.** Com base no último mês, cada execução reescalaria o histórico
inteiro; com base fixa a série só cresce no fim, os pontos já publicados não se mexem e o
primeiro ponto real é igual ao nominal do mesmo mês. O custo é que o `R$` do real é de março
de 2004 — a base agora está no nome/descrição do catálogo; o símbolo da unidade continua `R$`.

## Execução sem sobrescrever o publicado

A variável de ambiente `MVGR_SAIDA` redireciona **a pasta** dos dois arquivos de saída:

```
MVGR_SAIDA=/caminho/temporario jupyter nbconvert --execute --to notebook MVG-R.ipynb
```

## Coleta das séries oficiais

As duas séries são lidas desde o início, sem atualização incremental e sem ler o JSON já
publicado para decidir o período. A leitura repete com espera crescente porque o SGS devolve
502 esporadicamente.

In [1]:
from datetime import date
from pathlib import Path
import json
import math
import os
import time

import requests

RAIZ = Path(os.environ.get("DADOS_RAIZ", Path.cwd())).resolve()

def relativo(caminho):
    """Caminho exibido no notebook: relativo à raiz, nunca o absoluto da máquina."""
    try:
        return caminho.relative_to(RAIZ)
    except ValueError:
        return caminho.name
PASTA = RAIZ / "indicadores" / "mvgr" if (RAIZ / "indicadores" / "mvgr").is_dir() else RAIZ
DESTINO = Path(os.environ.get("MVGR_SAIDA", PASTA))
ARQUIVO_NOMINAL = DESTINO / "mvgr-nominal.json"
ARQUIVO_REAL = DESTINO / "mvgr-real.json"

SERIE_MVGR = 25419
SERIE_IPCA = 433
URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{serie}/dados"
URL_FONTE = URL.format(serie=SERIE_MVGR) + "?formato=json"
BASE = date(2004, 3, 1)
HOJE = date.today()


def baixar(serie, tentativas=4):
    """Baixa uma série SGS e devolve [(date, float)] em ordem ascendente."""
    for tentativa in range(tentativas):
        try:
            resposta = requests.get(
                URL.format(serie=serie), params={"formato": "json"}, timeout=60
            )
            resposta.raise_for_status()
            bruto = resposta.json()
            break
        except (requests.RequestException, ValueError):
            if tentativa == tentativas - 1:
                raise
            time.sleep(2 ** tentativa)
    if not bruto:
        raise ValueError(f"a série SGS {serie} não retornou dados")
    leituras = []
    for item in bruto:
        dia, mes, ano = (int(parte) for parte in item["data"].split("/"))
        leituras.append((date(ano, mes, dia), float(item["valor"])))
    leituras.sort()
    return leituras


def indice_mensal(dia):
    """Mês absoluto: serve para exigir um ponto por mês, sem lacuna."""
    return dia.year * 12 + (dia.month - 1)


def conferir_serie_mensal(nome, leituras):
    """Recusa série vazia, repetida, com lacuna, fora do dia 01 ou aberta no mês corrente."""
    if not leituras:
        raise ValueError(f"{nome}: série vazia")
    if len({dia for dia, _ in leituras}) != len(leituras):
        raise ValueError(f"{nome}: meses repetidos")
    if any(dia.day != 1 for dia, _ in leituras):
        raise ValueError(f"{nome}: data fora do dia 01")
    for (anterior, _), (seguinte, _) in zip(leituras, leituras[1:]):
        if indice_mensal(seguinte) - indice_mensal(anterior) != 1:
            raise ValueError(f"{nome}: lacuna entre {anterior} e {seguinte}")
    for dia, valor in leituras:
        if not math.isfinite(valor):
            raise ValueError(f"{nome}: valor não finito em {dia}")
    if leituras[-1][0] >= HOJE.replace(day=1):
        raise ValueError(f"{nome}: período corrente ainda aberto na fonte")


def brl(valor):
    """Formata em reais no padrão brasileiro (R$ 44.419,00)."""
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"R$ {texto}"


mvgr = baixar(SERIE_MVGR)
ipca_leituras = baixar(SERIE_IPCA)
conferir_serie_mensal(f"SGS {SERIE_MVGR}", mvgr)
conferir_serie_mensal(f"SGS {SERIE_IPCA}", ipca_leituras)
ipca = dict(ipca_leituras)

if mvgr[0][0] != BASE:
    raise ValueError(
        f"o início da série mudou para {mvgr[0][0]}; a base do deflator ({BASE}) precisa ser revista"
    )

faltando = [dia for dia, _ in mvgr if dia > BASE and dia not in ipca]
if faltando:
    raise ValueError(f"IPCA sem cobertura para {faltando[:3]}")

print(f"SGS {SERIE_MVGR} (MVG-R): {len(mvgr)} leituras, {mvgr[0][0]} a {mvgr[-1][0]}")
print(f"SGS {SERIE_IPCA} (IPCA mensal): {len(ipca_leituras)} leituras, "
      f"{ipca_leituras[0][0]} a {ipca_leituras[-1][0]}")
print(f"Base do deflator: {BASE}")
print(f"Pasta de saída: {relativo(DESTINO)}")
print(f"Mês corrente excluído: {HOJE:%Y-%m}")
print("\nPrimeiras 3 leituras do MVG-R:")
for dia, valor in mvgr[:3]:
    print(f"  {dia:%Y-%m-%d}  {brl(valor)}")
print("Últimas 3 leituras do MVG-R:")
for dia, valor in mvgr[-3:]:
    print(f"  {dia:%Y-%m-%d}  {brl(valor)}")

SGS 25419 (MVG-R): 270 leituras, 2004-03-01 a 2026-08-01
SGS 433 (IPCA mensal): 561 leituras, 1980-01-01 a 2026-09-01
Base do deflator: 2004-03-01
Pasta de saída: indicadores/mvgr
Mês corrente excluído: 2026-11

Primeiras 3 leituras do MVG-R:
  2004-03-01  R$ 44.419,00
  2004-04-01  R$ 43.000,00
  2004-05-01  R$ 43.200,00
Últimas 3 leituras do MVG-R:
  2026-06-01  R$ 285.000,00
  2026-07-01  R$ 286.000,00
  2026-08-01  R$ 290.000,00


## Série nominal

Sem cálculo: cada ponto é a leitura da SGS 25419 naquele mês. O arquivo é gravado inteiro —
nada de merge com o JSON já publicado —, com `versao`, `id`, `unidade`, `eixo`, `medida`,
`granularidade`, `fonte`, `urlFonte` e `dados` no formato `{"data": "YYYY-MM-DD", "valor": ...}`.

In [2]:
def numero(valor):
    """Mantém inteiro quando a fonte publica inteiro (a 25419 é em reais inteiros)."""
    return int(valor) if float(valor).is_integer() else float(valor)


serie_nominal = {
    "versao": 1,
    "id": "mvgr-nominal",
    "unidade": {"tipo": "moeda", "simbolo": "R$", "casas": 0},
    "eixo": "reais",
    "medida": "nivel",
    "granularidade": "mensal",
    "fonte": "Banco Central do Brasil (SGS 25419)",
    "urlFonte": URL_FONTE,
    "dados": [
        {"data": dia.strftime("%Y-%m-%d"), "valor": numero(valor)} for dia, valor in mvgr
    ],
}

DESTINO.mkdir(parents=True, exist_ok=True)
with ARQUIVO_NOMINAL.open("w", encoding="utf-8") as arquivo:
    json.dump(serie_nominal, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")

print(f"{len(serie_nominal['dados'])} pontos escritos em {ARQUIVO_NOMINAL}")
print(f"Primeiro ponto: {serie_nominal['dados'][0]}")
print(f"Último ponto:   {serie_nominal['dados'][-1]}")

270 pontos escritos em indicadores/mvgr/mvgr-nominal.json
Primeiro ponto: {'data': '2004-03-01', 'valor': 44419}
Último ponto:   {'data': '2026-08-01', 'valor': 290000}


## Série real — deflação pela média da janela de 3 meses

O deflator do mês `t` é o produto dos fatores `(1 + i/100)` da **média geométrica do IPCA da
janela de 3 meses** (`t-2`, `t-1`, `t`) de `2004-04` até `t`. A janela é a mesma do MVG-R
nominal, que é uma mediana móvel de 3 meses: deflacionar pelo IPCA do mês corrente trataria
uma mediana de três meses como se fosse uma observação pontual. Em `2004-03` o deflator é 1
por definição, então o primeiro ponto real é igual ao nominal. Cada mês entra **uma única
vez**: o valor real é `nominal / deflator`, sem soma de taxas e sem índice acumulado
interpolado.


In [3]:
def somar_meses(dia, n):
    """Desloca um mês absoluto, sempre no dia 01."""
    total = dia.year * 12 + dia.month - 1 + n
    return date(total // 12, total % 12 + 1, 1)


def media_ipca(dia):
    """Média geométrica do IPCA de M-2, M-1 e M (a janela móvel do MVG-R)."""
    fatores = [1 + ipca[somar_meses(dia, k)] / 100 for k in (-2, -1, 0)]
    return (math.prod(fatores) ** (1 / 3) - 1) * 100


deflator = {}
fator = 1.0
for dia, _ in mvgr:
    if dia > BASE:
        fator *= 1 + media_ipca(dia) / 100
    deflator[dia] = fator

serie_real = {
    "versao": 1,
    "id": "mvgr-real",
    "unidade": {"tipo": "moeda", "simbolo": "R$", "casas": 2},
    "eixo": "reais",
    "medida": "nivel",
    "granularidade": "mensal",
    "fonte": (
        "Banco Central do Brasil (SGS 25419); deflator IPCA/IBGE (SGS 433, "
        "média geométrica de 3 meses); cálculo do site"
    ),
    "urlFonte": URL_FONTE,
    "dados": [
        {"data": dia.strftime("%Y-%m-%d"), "valor": valor / deflator[dia]}
        for dia, valor in mvgr
    ],
}

with ARQUIVO_REAL.open("w", encoding="utf-8") as arquivo:
    json.dump(serie_real, arquivo, ensure_ascii=False, indent=2)
    arquivo.write("\n")

print(f"{len(serie_real['dados'])} pontos escritos em {ARQUIVO_REAL}")
print(f"Primeiro ponto: {serie_real['dados'][0]}")
print(f"Último ponto:   {serie_real['dados'][-1]}")
print("\nDeflator do IPCA (média de 3 meses; base 2004-03 = 1):")
for alvo in (date(2004, 3, 1), date(2008, 9, 1), date(2015, 10, 1), date(2016, 1, 1),
             date(2020, 4, 1), date(2025, 7, 1), mvgr[-1][0]):
    print(f"  {alvo:%Y-%m-%d}  {deflator[alvo]:8.6f}")

270 pontos escritos em indicadores/mvgr/mvgr-real.json
Primeiro ponto: {'data': '2004-03-01', 'valor': 44419.0}
Último ponto:   {'data': '2026-08-01', 'valor': 85662.07907256289}

Deflator do IPCA (média de 3 meses; base 2004-03 = 1):
  2004-03-01  1.000000
  2008-09-01  1.263332
  2015-10-01  1.936295
  2016-01-01  1.991009
  2020-04-01  2.364592
  2025-07-01  3.237400
  2026-08-01  3.385395


## Validação inline

A célula final falha se o contrato, a ordem, a unicidade e a continuidade dos meses, a
finitude dos valores, a igualdade com a fonte ou a coerência entre nominal e real forem
violadas. Ela confere em especial que **o fator entre dois meses consecutivos é exatamente a
média geométrica do IPCA da janela de 3 meses** — é essa verificação que pega o erro de
aplicar a mesma média duas vezes.


In [4]:
CHAVES_RAIZ = {
    "versao", "id", "unidade", "eixo", "medida", "granularidade", "fonte", "urlFonte", "dados",
}
UNIDADE_NOMINAL = {"tipo": "moeda", "simbolo": "R$", "casas": 0}
UNIDADE_REAL = {"tipo": "moeda", "simbolo": "R$", "casas": 2}
FONTE_NOMINAL = "Banco Central do Brasil (SGS 25419)"
FONTE_REAL = (
    "Banco Central do Brasil (SGS 25419); deflator IPCA/IBGE (SGS 433, "
    "média geométrica de 3 meses); cálculo do site"
)

for serie, identificador, unidade, fonte in (
    (serie_nominal, "mvgr-nominal", UNIDADE_NOMINAL, FONTE_NOMINAL),
    (serie_real, "mvgr-real", UNIDADE_REAL, FONTE_REAL),
):
    assert set(serie) == CHAVES_RAIZ, sorted(set(serie) ^ CHAVES_RAIZ)
    assert serie["versao"] == 1, serie["versao"]
    assert serie["id"] == identificador, serie["id"]
    assert serie["unidade"] == unidade, serie["unidade"]
    assert serie["eixo"] == "reais", serie["eixo"]
    assert serie["medida"] == "nivel", serie["medida"]
    assert serie["granularidade"] == "mensal", serie["granularidade"]
    assert serie["fonte"] == fonte, serie["fonte"]
    assert serie["urlFonte"].startswith("https://"), serie["urlFonte"]

    dados = serie["dados"]
    assert dados, "série vazia"
    datas = [ponto["data"] for ponto in dados]
    dias = [date.fromisoformat(texto) for texto in datas]
    assert all(dia.day == 1 for dia in dias), "dia diferente de 01"
    assert all(a < b for a, b in zip(dias, dias[1:])), "datas fora de ordem ou repetidas"
    assert all(
        indice_mensal(b) - indice_mensal(a) == 1 for a, b in zip(dias, dias[1:])
    ), "mês faltando ou repetido"
    for ponto in dados:
        assert set(ponto) == {"data", "valor"}, sorted(ponto)
        assert not isinstance(ponto["valor"], bool), ponto
        assert isinstance(ponto["valor"], (int, float)), ponto
        assert math.isfinite(ponto["valor"]), ponto

    assert len(dados) == len(mvgr), (len(dados), len(mvgr))
    assert dias[0] == mvgr[0][0] and dias[-1] == mvgr[-1][0], (dias[0], dias[-1])
    assert dias[-1] < HOJE.replace(day=1), dias[-1]

    with (ARQUIVO_NOMINAL if identificador == "mvgr-nominal" else ARQUIVO_REAL).open(
        encoding="utf-8"
    ) as arquivo:
        assert json.load(arquivo) == serie, "arquivo gravado difere do objeto validado"

# 1) Nominal idêntico à fonte, ponto a ponto.
esperado_nominal = [
    {"data": dia.strftime("%Y-%m-%d"), "valor": numero(valor)} for dia, valor in mvgr
]
assert serie_nominal["dados"] == esperado_nominal, "nominal difere da SGS 25419"

# 2) Real = nominal / deflator, em todo mês (a relação exigida é exata).
nominal_por_data = {ponto["data"]: ponto["valor"] for ponto in serie_nominal["dados"]}
for ponto in serie_real["dados"]:
    dia = date.fromisoformat(ponto["data"])
    assert abs(ponto["valor"] - nominal_por_data[ponto["data"]] / deflator[dia]) < 1e-9, ponto
    assert abs(ponto["valor"] / nominal_por_data[ponto["data"]] - 1 / deflator[dia]) < 1e-12, ponto

# 3) O deflator avança exatamente uma média da janela de 3 meses por mês — nunca duas.
for (anterior, _), (seguinte, _) in zip(mvgr, mvgr[1:]):
    crescimento = deflator[seguinte] / deflator[anterior]
    esperado = 1 + media_ipca(seguinte) / 100
    assert abs(crescimento - esperado) < 1e-12, (seguinte, crescimento, esperado)

# 4) O primeiro ponto real é o nominal do mês-base.
assert deflator[BASE] == 1.0, deflator[BASE]
assert abs(serie_real["dados"][0]["valor"] - serie_nominal["dados"][0]["valor"]) < 1e-12

# 5) Os dois meses de out/2005 e out/2015 entram uma única vez.
posicao = {dia: i for i, (dia, _) in enumerate(mvgr)}
for alvo in (date(2005, 10, 1), date(2015, 10, 1)):
    i = posicao[alvo]
    crescimento = deflator[alvo] / deflator[mvgr[i - 1][0]]
    esperado = 1 + media_ipca(alvo) / 100
    print(
        f"{alvo:%Y-%m}: fator {crescimento:.6f} = 1 + média "
        f"{media_ipca(alvo):.4f}% (não {esperado ** 2:.6f})"
    )
    assert abs(crescimento - esperado) < 1e-12, alvo

# 6) Números citados em docs/metodologia.md.
assert serie_nominal["dados"][0] == {"data": "2004-03-01", "valor": 44419}
assert nominal_por_data["2025-07-01"] == 260000, nominal_por_data["2025-07-01"]
assert abs(serie_real["dados"][-1]["valor"] - 290000 / deflator[mvgr[-1][0]]) < 1e-9

print(
    f"ok: mvgr-nominal e mvgr-real com {len(mvgr)} meses únicos, "
    f"{mvgr[0][0]} a {mvgr[-1][0]}, nominal idêntico à SGS 25419 e real = nominal / IPCA "
    f"(base {BASE:%Y-%m})"
)

2005-10: fator 1.004230 = 1 + média 0.4230% (não 1.008479)
2015-10: fator 1.005264 = 1 + média 0.5264% (não 1.010555)
ok: mvgr-nominal e mvgr-real com 270 meses únicos, 2004-03-01 a 2026-08-01, nominal idêntico à SGS 25419 e real = nominal / IPCA (base 2004-03)
